In [ ]:
from pathlib import Path
import pandas as pd

"""
Special-case correction for Sonata 25 motif annotations.

This script reconstructs motif labels for Sonata 25 from the motif note files
and merges them back into the main CSV note table. It was used to produce the
working representation employed in the paper's analysis pipeline.

Base data source: BPS-Motif / Beethoven motif annotation files.
"""

# Repository root
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

# Paths
csv_main_path = ROOT / "data" / "raw" / "BPS_Motif" / "csv_notes" / "25-1.csv"
motif_folder_path = ROOT / "data" / "raw" / "BPS_Motif" / "motif_notes" / "25"
output_path = ROOT / "data" / "processed" / "BPS_Motif_corrected" / "25-1_corrected.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)

# Expected column names
column_names = ["onset", "midi_number", "morphetic_number", "duration", "staff_number", "measure"]
merge_keys = ["onset", "midi_number", "morphetic_number", "duration", "staff_number", "measure"]

# Load main CSV file
df_main = pd.read_csv(csv_main_path)
df_main["type"] = None  # Reset existing type values before rebuilding labels
df_main[column_names] = df_main[column_names].astype(float)

print("\nMain DataFrame Head:")
print(df_main.head())

# Dictionary to store motif DataFrames grouped by motif label
motif_data = {}

# Process all motif files
for file_path in sorted(motif_folder_path.glob("*.csv")):
    motif_label = file_path.stem.split("-")[0].lower()

    df_motif = pd.read_csv(
        file_path,
        delimiter=",",
        header=None,
        names=column_names
    )

    df_motif[column_names] = df_motif[column_names].astype(float)
    df_motif["type"] = motif_label

    if motif_label in motif_data:
        motif_data[motif_label] = pd.concat([motif_data[motif_label], df_motif], ignore_index=True)
    else:
        motif_data[motif_label] = df_motif

# Concatenate all motif DataFrames
df_motifs = pd.concat(motif_data.values(), ignore_index=True)
df_motifs = df_motifs.sort_values(by=["onset"])

print("\nMotif DataFrame Head (After Fixing Grouping):")
print(df_motifs.head())

# Merge corrected motif labels back into the main table
df_merged = df_main.merge(df_motifs, on=merge_keys, how="left", suffixes=("", "_motif"))
df_merged["type"] = df_merged["type_motif"]
df_merged.drop(columns=["type_motif"], inplace=True)

print("\nMerged DataFrame (After Fixing):")
print(df_merged.head(20))

# Save corrected output
df_merged.to_csv(output_path, index=False)
print(f"\nSaved corrected file to: {output_path}")